In [1]:
import pulp as plp, pandas as pd
from math import radians, sin, cos, sqrt, atan2
solver = plp.GLPK_CMD()

df_villes = pd.read_csv("../data/villes.csv", index_col=0)
df_villes.head()

,latitude,longitude,demande
ville,,,
Paris,48.8566,2.3522,0
Lyon,45.7640,4.8357,20
Marseille,43.2965,5.3698,25
Toulouse,43.6047,1.4442,15
Bordeaux,44.8378,-0.5792,30


In [2]:
villes = list(df_villes.index)
nb_villes = len(villes)

In [3]:
nb_vehicules = 3
capacite_max = 100
depot = "Paris"
demande = df_villes['demande'].to_dict()
clients = [v for v in villes if v!=depot]

In [4]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = radians(lat2-lat1)
    dlon = radians(lon2-lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    return 2*R*atan2(sqrt(a), sqrt(1-a))

In [5]:
distances = {}
for i in villes:
    for j in villes:
        if i != j:
            distances[i, j] = haversine(df_villes.loc[i, 'latitude'], df_villes.loc[i, 'longitude'], 
                                        df_villes.loc[j, 'latitude'], df_villes.loc[j, 'longitude'])

In [6]:
model = plp.LpProblem('Vehicule_Routing_Problem', plp.LpMinimize)

# Variables de décision
x = plp.LpVariable.dicts('x', [(i, j, k) for i in villes for j in villes for k in range(1, nb_vehicules+1)], cat = "Binary")
u = plp.LpVariable.dicts('u', clients, lowBound = 1,upBound = nb_villes - 1, cat='Integers')

# Fonction objectif
model += plp.lpSum(distances[i, j]*x[i, j, k] for i in villes for j in villes for k in range(1, nb_vehicules+1) if i!=j)

# Contraintes
for j in clients:
    model += plp.lpSum(x[i, j, k] for i in villes for k in range(1, nb_vehicules+1) if i!= j) == 1 

for i in clients:
    model += plp.lpSum(x[i, j, k] for j in villes for k in range(1, nb_vehicules+1) if i!= j) == 1 

for k in range(1, nb_vehicules+1):
    model += plp.lpSum(x[i, depot, k] for i in clients) == 1  
    model += plp.lpSum(x[depot, j, k] for j in clients) == 1  
    model += plp.lpSum(demande[i]*x[i, j, k] for i in clients for j in villes if i!=j) <= capacite_max  
    for h in clients:
        model += (plp.lpSum(x[i, h, k] for i in villes if i != h) == 
                  plp.lpSum(x[h, j, k] for j in villes if j != h))
    

In [7]:
for i in clients:
    for j in clients:
        for k in range(1, nb_vehicules+1):
             if i!=j:
                 model += u[i] - u[j] + nb_villes*x[i, j, k] <= nb_villes - 1


# Résolution
model.solve(solver)

GLPSOL--GLPK LP/MIP Solver 5.0
Parameter(s) specified in the command line:
 --cpxlp /tmp/af7b6c96edfe4a9f8d9be5a39c7f818e-pulp.lp -o /tmp/af7b6c96edfe4a9f8d9be5a39c7f818e-pulp.out
 -w /tmp/af7b6c96edfe4a9f8d9be5a39c7f818e-pulp.sol
Reading problem data from '/tmp/af7b6c96edfe4a9f8d9be5a39c7f818e-pulp.lp'...
329 rows, 340 columns, 2370 non-zeros
330 integer variables, all of which are binary
1739 lines were read
GLPK Integer Optimizer 5.0
329 rows, 340 columns, 2370 non-zeros
330 integer variables, all of which are binary
Preprocessing...
270 constraint coefficient(s) were reduced
329 rows, 340 columns, 2370 non-zeros
330 integer variables, all of which are binary
Scaling...
 A: min|aij| =  1.000e+00  max|aij| =  3.000e+01  ratio =  3.000e+01
GM: min|aij| =  7.488e-01  max|aij| =  1.335e+00  ratio =  1.783e+00
EQ: min|aij| =  5.616e-01  max|aij| =  1.000e+00  ratio =  1.781e+00
2N: min|aij| =  4.688e-01  max|aij| =  1.250e+00  ratio =  2.667e+00
Constructing initial basis...
Size of tria

1

In [8]:
# Résultats
print(f"Statut : {plp.LpStatus[model.status]}")
print(f"Distance totale : {plp.value(model.objective):.0f} km\n")

# Reconstruire les tournées par véhicule
for k in range(1, 4):
    tournee = [depot]
    ville_actuelle = depot
    distance_k = 0

    while True:
        next_ville = None
        for j in villes:
            if ville_actuelle != j and plp.value(x[ville_actuelle, j, k]) == 1:
                next_ville = j
                break
        if next_ville is None:
            break
        distance_k += distances[ville_actuelle, next_ville]
        tournee.append(next_ville)
        ville_actuelle = next_ville
        if next_ville == depot:
            break

    if len(tournee) > 2:
        print(f"Véhicule {k} — Distance : {distance_k:.0f} km")
        print(f"  {' → '.join(tournee)}")
        charge = sum(demande[v] for v in tournee if v != depot)
        print(f"  Charge : {charge}/{capacite_max} unités\n")

Statut : Optimal
Distance totale : 3675 km

Véhicule 1 — Distance : 1784 km
  Paris → Strasbourg → Grenoble → Marseille → Montpellier → Lyon → Paris
  Charge : 88/100 unités

Véhicule 2 — Distance : 407 km
  Paris → Lille → Paris
  Charge : 25/100 unités

Véhicule 3 — Distance : 1483 km
  Paris → Rennes → Nantes → Bordeaux → Toulouse → Paris
  Charge : 85/100 unités

